# Resumo do código

Este notebook implementa um fluxo completo para **gerar** e **verificar** hash SHA-256 de arquivos.

Ele permite:
- Ler um arquivo qualquer.
- Calcular sua chave hash SHA-256.
- Mostrar essa chave na saída.
- Salvar a chave em um novo arquivo com o mesmo nome base e extensão diferente (ex.: `.sha256`).
- Ler novamente um arquivo e um arquivo de hash para verificar se o conteúdo corresponde àquela chave.

## Estrutura das células

1. **Importações e configuração**: importa bibliotecas e define extensão padrão para arquivo de hash.
2. **Funções de geração e salvamento**: calcula o SHA-256 de um arquivo e salva a chave em disco.
3. **Funções de leitura e verificação**: lê hash salvo e valida se um arquivo corresponde ao hash informado.
4. **Exemplo de geração de hash**: mostra como gerar e salvar hash para um arquivo.
5. **Exemplo de verificação**: mostra como comparar um arquivo com um hash salvo.

In [2]:
# Esta célula importa as bibliotecas necessárias e define configurações padrão.
import hashlib
from pathlib import Path

EXTENSAO_HASH_PADRAO = ".sha256"

In [3]:
# Esta célula define funções para gerar o hash SHA-256 e salvar em um novo arquivo.
def gerar_hash_sha256(caminho_arquivo: str | Path, tamanho_bloco: int = 65536) -> str:
    caminho = Path(caminho_arquivo)
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    sha256 = hashlib.sha256()
    with caminho.open("rb") as arquivo:
        while True:
            bloco = arquivo.read(tamanho_bloco)
            if not bloco:
                break
            sha256.update(bloco)

    return sha256.hexdigest()


def salvar_hash_em_arquivo(caminho_arquivo: str | Path, hash_hex: str, extensao_hash: str = EXTENSAO_HASH_PADRAO) -> Path:
    caminho = Path(caminho_arquivo)
    caminho_hash = caminho.with_suffix(extensao_hash)
    caminho_hash.write_text(hash_hex + "\n", encoding="utf-8")
    return caminho_hash

In [4]:
# Esta célula define funções para ler um hash salvo e verificar se um arquivo corresponde a ele.
def ler_hash_de_arquivo(caminho_hash: str | Path) -> str:
    caminho = Path(caminho_hash)
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo de hash não encontrado: {caminho}")

    hash_lido = caminho.read_text(encoding="utf-8").strip().lower()
    if len(hash_lido) != 64 or any(c not in "0123456789abcdef" for c in hash_lido):
        raise ValueError("Conteúdo do arquivo de hash inválido para SHA-256.")

    return hash_lido


def verificar_arquivo_com_hash(caminho_arquivo: str | Path, caminho_hash: str | Path) -> bool:
    hash_calculado = gerar_hash_sha256(caminho_arquivo)
    hash_esperado = ler_hash_de_arquivo(caminho_hash)
    return hash_calculado == hash_esperado

In [8]:
# Esta célula mostra como gerar o hash de um arquivo, exibir na tela e salvar em arquivo .sha256.
caminho_arquivo_entrada = "LenaC.jpg"  # Altere para o arquivo que deseja processar.

hash_gerado = gerar_hash_sha256(caminho_arquivo_entrada)
caminho_hash_gerado = salvar_hash_em_arquivo(caminho_arquivo_entrada, hash_gerado)

print(f"Arquivo lido: {caminho_arquivo_entrada}")
print(f"Hash SHA-256: {hash_gerado}")
print(f"Arquivo de hash salvo em: {caminho_hash_gerado}")

Arquivo lido: LenaC.jpg
Hash SHA-256: 27788e233759866b259a166d869e085e9fb930de1c1c5c0509f3078a8589b252
Arquivo de hash salvo em: LenaC.sha256


In [12]:
# Esta célula mostra como verificar se um novo arquivo corresponde a um hash salvo.
caminho_novo_arquivo = "LenaC.jpg"  # Altere para o arquivo que deseja verificar.
caminho_arquivo_hash = "LenaC.sha256"  # Altere para o arquivo de hash correspondente.

arquivo_valido = verificar_arquivo_com_hash(caminho_novo_arquivo, caminho_arquivo_hash)

if arquivo_valido:
    print("Verificação bem-sucedida: o arquivo corresponde ao hash informado.")
else:
    print("Falha na verificação: o arquivo NÃO corresponde ao hash informado.")

Falha na verificação: o arquivo NÃO corresponde ao hash informado.
